In [3]:
import ConnectionConfig as cc
from pyspark.sql.functions import *
cc.setupEnvironment()
cc.listEnvironment()

ALLUSERSPROFILE: C:\ProgramData
APPDATA: C:\Users\lzkol\AppData\Roaming
COMMONPROGRAMFILES: C:\Program Files\Common Files
COMMONPROGRAMFILES(X86): C:\Program Files (x86)\Common Files
COMMONPROGRAMW6432: C:\Program Files\Common Files
COMPUTERNAME: LIZA
COMSPEC: C:\WINDOWS\system32\cmd.exe
DRIVERDATA: C:\Windows\System32\Drivers\DriverData
EFC_13296_1262719628: 1
EFC_13296_1592913036: 1
EFC_13296_2283032206: 1
EFC_13296_2775293581: 1
EFC_13296_3789132940: 1
FPS_BROWSER_APP_PROFILE_STRING: Internet Explorer
FPS_BROWSER_USER_PROFILE_STRING: Default
GOPATH: C:\Users\lzkol\go
HOMEDRIVE: C:
HOMEPATH: \Users\lzkol
IPY_INTERRUPT_EVENT: 1748
JAVA_HOME: C:\Program Files\Java\jdk-11\
JPY_INTERRUPT_EVENT: 1748
JPY_PARENT_PID: 840
JPY_SESSION_NAME: dimVehicle.ipynb
LANG: en_US.UTF-8
LANGUAGE: 
LC_ALL: en_US.UTF-8
LOCALAPPDATA: C:\Users\lzkol\AppData\Local
LOGONSERVER: \\LIZA
NUMBER_OF_PROCESSORS: 16
ONEDRIVE: C:\Users\lzkol\OneDrive
OS: Windows_NT
PATH: C:\Users\lzkol\PycharmProjects\sparkdelta\.ven

In [4]:
spark = cc.startLocalCluster("dimVehicle")
spark.getActiveSession()

In [6]:
# EXTRACT
cc.set_connectionProfile("velodb")

df_vehicles = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "vehicles") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_bikelots = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "bikelots") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_biketypes = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "bike_types") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

In [7]:
print(cc.create_jdbc())

jdbc:postgresql://localhost:5432/velodb?user=postgres&password=Student_1234&ssl=false


In [8]:
# TRANSFORM: join vehicles → bikelots → bike_types
df_vehicle_dim = df_vehicles.alias("v") \
    .join(df_bikelots.alias("b"), col("v.bikelotid") == col("b.bikelotid"), "left") \
    .join(df_biketypes.alias("t"), col("b.biketypeid") == col("t.biketypeid"), "left") \
    .select(
        col("v.vehicleid").alias("vehicle_id"),
        col("t.biketypedescription").alias("type")
    )

In [10]:
# Inspect schema and preview
df_vehicle_dim.show()
df_vehicle_dim.printSchema()

# LOAD: write as delta table
df_vehicle_dim.write.format("delta").mode("overwrite").saveAsTable("dimvehicle")

+----------+---------+
|vehicle_id|     type|
+----------+---------+
|         1|Velo Bike|
|         2|Velo Bike|
|         3|Velo Bike|
|         4|Velo Bike|
|         5|Velo Bike|
|         6|Velo Bike|
|         7|Velo Bike|
|         8|Velo Bike|
|         9|Velo Bike|
|        10|Velo Bike|
|        11|Velo Bike|
|        12|Velo Bike|
|        13|Velo Bike|
|        14|Velo Bike|
|        15|Velo Bike|
|        16|Velo Bike|
|        17|Velo Bike|
|        18|Velo Bike|
|        19|Velo Bike|
|        20|Velo Bike|
+----------+---------+
only showing top 20 rows

root
 |-- vehicle_id: integer (nullable = true)
 |-- type: string (nullable = true)



## Delete the spark session

In [9]:
spark.stop()